In [1]:
%cd ..

/home/ricka/Git/GitHub/RickArko/kaggle/playground/kcom-student-health-risk


# Student Health Risk — TabPFN v3

**TabPFN** is a foundation model for tabular data — it uses in-context learning
via a transformer trained on millions of synthetic datasets.  No hyperparameter
tuning, no feature engineering, no imputation needed.

[TabPFN on GitHub](https://github.com/priorlabs/tabpfn) |
[Paper](https://arxiv.org/abs/2407.13271)

> **Idea**: Let a foundation model (trained on 10M+ synthetic tabular tasks)
> classify student health risk.  TabPFN handles missing values, categoricals,
> and class imbalance without any manual preprocessing.

## 0. TabPFN License & HuggingFace Access

**TabPFN v3 requires two credentials.**

### Prior Labs API Key
1. Open https://ux.priorlabs.ai in a browser and log in (or register)
2. Accept the license on the **Licenses** tab
3. Copy your **API Key** from https://ux.priorlabs.ai/account
4. Set `TABPFN_TOKEN` in `.env`: `TABPFN_TOKEN=<your-api-key>`

### HuggingFace Token (gated model weights)
1. Create a read token at https://huggingface.co/settings/tokens
2. Accept the terms at https://huggingface.co/Prior-Labs/tabpfn_3
3. Set `HF_TOKEN` in `.env`: `HF_TOKEN=hf_<your-token>`

Both tokens should be placed in a `.env` file in the project root.

## 1. Setup

In [2]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
from tabpfn import TabPFNClassifier

warnings.filterwarnings("ignore")

# ── Load credentials from .env ──
load_dotenv()  # .env -> os.environ

_TABPFN_TOKEN = os.environ.get("TABPFN_TOKEN")
if _TABPFN_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        _TABPFN_TOKEN = UserSecretsClient().get_secret("TABPFN_TOKEN")
    except Exception:
        pass
if _TABPFN_TOKEN is None:
    _TABPFN_TOKEN = os.environ.get("KAGGLE_SECRET_TABPFN_TOKEN")
if _TABPFN_TOKEN is not None:
    os.environ["TABPFN_TOKEN"] = _TABPFN_TOKEN
    print("✓ TABPFN_TOKEN loaded")
else:
    print("✗ TABPFN_TOKEN not set — see license cell above")

# TabPFN v3 weights live on a gated HuggingFace repo — needs HF_TOKEN too
_HF_TOKEN = os.environ.get("HF_TOKEN")
if _HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        _HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if _HF_TOKEN is not None:
    os.environ["HF_TOKEN"] = _HF_TOKEN
    print("✓ HF_TOKEN loaded")
else:
    print("✗ HF_TOKEN not set — TabPFN v3 may fail to download weights")

NUM_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
CAT_COLS = [
    "diet_type", "stress_level", "sleep_quality",
    "physical_activity_level", "smoking_alcohol", "gender",
]
TARGET = "health_condition"
TARGET_MAP = {"fit": 0, "at-risk": 1, "unhealthy": 2}

_DATA_CANDIDATES = [Path("data/raw"), Path("../data/raw")]
DATA_DIR = next((p for p in _DATA_CANDIDATES if (p / "train.csv").exists()), _DATA_CANDIDATES[0])
print(f"✓ DATA_DIR = {DATA_DIR.resolve()}")

✓ TABPFN_TOKEN loaded
✓ HF_TOKEN loaded
✓ DATA_DIR = /home/ricka/Git/GitHub/RickArko/kaggle/playground/kcom-student-health-risk/data/raw


## 2. Load Data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_ids = test["id"].copy()

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Target:\n{train[TARGET].value_counts()}")

Train: (690088, 15)  |  Test: (295753, 14)
Target:
health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64


## 3. Prepare Features

TabPFN handles **missing values natively** — no imputation.
Categoricals are passed as strings; TabPFN encodes them internally.

We **stratified subsample** to 15K rows because TabPFN's in-context learning
sweet spot is 1K–15K samples.  The subsample preserves class proportions.

In [4]:
SAMPLE_SIZE = 15000
sss = StratifiedShuffleSplit(n_splits=1, train_size=SAMPLE_SIZE,
                              random_state=42)
idx, _ = next(sss.split(train, train[TARGET]))
sample = train.iloc[idx].reset_index(drop=True)

print(f"Sampled {len(sample):,} rows")
print(f"Distribution:\n{sample[TARGET].value_counts()}")

feature_cols = NUM_COLS + CAT_COLS
X_tr = sample[feature_cols].copy()
y_tr = sample[TARGET].map(TARGET_MAP).values

X_te = test[feature_cols].copy()

# Ensure dtypes
for c in NUM_COLS:
    X_tr[c] = X_tr[c].astype("float32")
    X_te[c] = X_te[c].astype("float32")
for c in CAT_COLS:
    X_tr[c] = X_tr[c].astype(str)
    X_te[c] = X_te[c].astype(str)

cat_indices = list(range(len(NUM_COLS), len(feature_cols)))
print(f"Categorical column indices: {cat_indices}")
print(f"X_tr: {X_tr.shape}  X_te: {X_te.shape}")

Sampled 15,000 rows
Distribution:
health_condition
at-risk      12880
unhealthy     1255
fit            865
Name: count, dtype: int64
Categorical column indices: [7, 8, 9, 10, 11, 12]
X_tr: (15000, 13)  X_te: (295753, 13)


## 4. Train TabPFN

TabPFN is a transformer — it "trains" by simply encoding the training data
as context.  The forward pass runs on GPU if available.

In [6]:
print("Fitting TabPFN ...")
model = TabPFNClassifier(
    categorical_features_indices=cat_indices,
    n_estimators=8,
    random_state=42,
    show_progress_bar=False,
)
model.fit(X_tr, y_tr)
print("✓ Fit complete")

Fitting TabPFN ...


tabpfn-v3-classifier-v3_default.ckpt:   0%|          | 0.00/213M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

✓ Fit complete


## 5. Predict & Submit

In [7]:
preds = model.predict_proba(X_te)
test_labels = preds.argmax(axis=1)
rev_map = {v: k for k, v in TARGET_MAP.items()}
pred_labels = [rev_map[i] for i in test_labels]

print("Predicted distribution:")
for label in ["fit", "at-risk", "unhealthy"]:
    pct = 100 * pred_labels.count(label) / len(pred_labels)
    print(f"  {label}: {pred_labels.count(label)} ({pct:.1f}%)")

Predicted distribution:
  fit: 14571 (4.9%)
  at-risk: 261531 (88.4%)
  unhealthy: 19651 (6.6%)


In [8]:
out_dir = Path("data/submissions")
out_dir.mkdir(parents=True, exist_ok=True)
submission = pd.DataFrame({"id": test_ids, "health_condition": pred_labels})
submission.to_csv(out_dir / "submission.csv", index=False)
print(f"Saved to {out_dir / 'submission.csv'}")
print(f"Shape: {submission.shape}")
submission.head()

Saved to data/submissions/submission.csv
Shape: (295753, 2)


,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 6. Feature Importance (TabPFN explanation)

TabPFN doesn't provide traditional feature importance, but we can estimate
it by measuring the drop in balanced accuracy when shuffling each feature.

In [9]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_tr)
base_score = balanced_accuracy_score(y_tr, y_pred)
print(f"Baseline balanced acc (in-sample): {base_score:.4f}")

importance = {}
for i, col in enumerate(feature_cols):
    X_perm = X_tr.copy()
    X_perm.iloc[:, i] = np.random.permutation(X_perm.iloc[:, i].values)
    y_perm = model.predict(X_perm)
    drop = base_score - balanced_accuracy_score(y_tr, y_perm)
    importance[col] = drop

imp_df = pd.Series(importance).sort_values(ascending=False)
print("\nFeature importance (drop in balanced acc when shuffled):")
for col, val in imp_df.items():
    bar = "█" * int(abs(val) * 200)
    print(f"  {col:25s} {val:+.4f}  {bar}")

Baseline balanced acc (in-sample): 0.8765

Feature importance (drop in balanced acc when shuffled):
  stress_level              +0.4073  █████████████████████████████████████████████████████████████████████████████████
  sleep_duration            +0.3414  ████████████████████████████████████████████████████████████████████
  physical_activity_level   +0.1913  ██████████████████████████████████████
  bmi                       +0.0233  ████
  step_count                +0.0110  ██
  smoking_alcohol           +0.0103  ██
  exercise_duration         +0.0091  █
  sleep_quality             +0.0071  █
  gender                    +0.0054  █
  water_intake              +0.0049  
  calorie_expenditure       +0.0046  
  heart_rate                +0.0045  
  diet_type                 +0.0042  


## 7. Summary

### What TabPFN brings

1. **Zero preprocessing** — no imputation, no encoding, no scaling
2. **No tuning** — foundation model works out of the box
3. **Strong baseline** — often beats tuned tree ensembles on small–medium data
4. **GPU accelerated** — runs on Kaggle's T4 GPU

### Limitations

- 15K subsample means we discard 98% of training data
- No native feature importance (permutation-based is slow)
- Larger ensemble (e.g., `n_estimators=32`) improves score but costs more

### Next steps to improve

1. Ensemble TabPFN with LightGBM (different inductive biases)
2. Multiple subsample runs averaged (reduce variance)
3. Increase `n_estimators` to 16–32 if GPU memory allows
4. Try `ignore_pretraining_limits=True` with more samples

In [10]:
print("✓ TabPFN baseline complete")
print(f"  Submission: data/submissions/submission.csv")

✓ TabPFN baseline complete
  Submission: data/submissions/submission.csv
